In [50]:
import sys
sys.path.append('.')  # para que encuentre fetcher.py
import importlib
import fetcher

from fetcher import DataFetcher
from grid_intelligence.params import ENTSOE_API_KEY

importlib.reload(fetcher)

from fetcher import DataFetcher

data_fetcher = DataFetcher(entsoe_api_key=ENTSOE_API_KEY, output_path='raw_data')

# Full fetch
data_fetcher.fetch_full(start='2026-04-18', end='2026-04-20')
print('Full fetch done')

Full fetch from 2026-04-18 to 2026-04-20...
  → ENTSO-E prices...
  → ENTSO-E generation...
  → ENTSO-E load...
  → ENTSO-E wind...
  → Open-Meteo weather...
  → Yahoo Finance TTF gas...
Saved: raw_data/consolidated_full.csv — 193 rows
Full fetch done


In [51]:
print(df.isnull().sum())

price                        0
generation                   2
consumption                  2
wind_onshore                 2
temperature_c                0
humidity_percent             0
cloud_cover_percent          0
shortwave_radiation_wm2      0
wind_speed_ms                0
ttf_gas                    289
dtype: int64


In [52]:
data_fetcher.fetch_delta()
print('Delta done')

Delta fetch from 2026-04-17 to 2026-04-23...
  → ENTSO-E prices...
  → ENTSO-E generation...
  → ENTSO-E load...
  → ENTSO-E wind...
  → Open-Meteo weather...
  → Yahoo Finance TTF gas...
Updated: raw_data/consolidated_full.csv — 577 rows
Delta done


In [53]:
df = pd.read_csv('raw_data/consolidated_full.csv', index_col=0, parse_dates=True)
print(df.index.duplicated().sum())  # sollte 0 sein

0


In [48]:
df.shape

(770, 10)

In [43]:
import pandas as pd

df = pd.read_csv('raw_data/consolidated_full.csv', index_col=0, parse_dates=True)

print(f"Vorher: {df.shape[0]} Zeilen, {df.index.duplicated().sum()} Duplikate")

df = df.sort_index()
df = df[~df.index.duplicated(keep='last')]

print(f"Nachher: {df.shape[0]} Zeilen, {df.index.duplicated().sum()} Duplikate")

df.to_csv('raw_data/consolidated_full.csv')
print("Gespeichert ✓")

Vorher: 674 Zeilen, 193 Duplikate
Nachher: 481 Zeilen, 0 Duplikate
Gespeichert ✓


In [55]:
import pandas as pd

df = pd.read_csv("raw_data/consolidated_full.csv", index_col=0, parse_dates=True)

print("Shape:", df.shape)
print("Von:", df.index.min())
print("Bis:", df.index.max())
print("\nFehlende Werte:")
print(df.isnull().sum())
print("\nErste Zeilen:")
print(df.head())

Shape: (264873, 10)
Von: 2018-09-30 22:00:00+00:00
Bis: 2026-04-21 00:00:00+00:00

Fehlende Werte:
price                      184176
generation                      1
consumption                    17
wind_onshore                    2
temperature_c               86792
humidity_percent            86792
cloud_cover_percent         86796
shortwave_radiation_wm2     86796
wind_speed_ms               86792
ttf_gas                      6122
dtype: int64

Erste Zeilen:
                           price  generation  consumption  wind_onshore  \
datetime_utc                                                              
2018-09-30 22:00:00+00:00  59.53    51434.81          NaN       4292.89   
2018-09-30 22:15:00+00:00    NaN    52085.57          NaN       4239.07   
2018-09-30 22:30:00+00:00    NaN    52345.33          NaN       4208.44   
2018-09-30 22:45:00+00:00    NaN    52902.03          NaN       4153.52   
2018-09-30 23:00:00+00:00  56.10    52799.83     42354.46       4047.26   

       

In [ ]:
df = pd.read_csv('raw_data/consolidated_full.csv', index_col=0, parse_dates=True)
print(df.index.duplicated().sum())  # sollte 0 sein

0


In [48]:
df.shape

(770, 10)

In [43]:
import pandas as pd

df = pd.read_csv('raw_data/consolidated_full.csv', index_col=0, parse_dates=True)

print(f"Vorher: {df.shape[0]} Zeilen, {df.index.duplicated().sum()} Duplikate")

df = df.sort_index()
df = df[~df.index.duplicated(keep='last')]

print(f"Nachher: {df.shape[0]} Zeilen, {df.index.duplicated().sum()} Duplikate")

df.to_csv('raw_data/consolidated_full.csv')
print("Gespeichert ✓")

Vorher: 674 Zeilen, 193 Duplikate
Nachher: 481 Zeilen, 0 Duplikate
Gespeichert ✓


In [6]:
import pandas as pd

df = pd.read_csv("raw_data/combined_energy_price_clean.csv",
                 sep='\t', low_memory=False)

# Nur DE-LU Sequence 2 PT15M
df_seq2 = df[
    (df['AreaDisplayName'] == 'DE-LU') &
    (df['ResolutionCode'] == 'PT15M')
].copy()

# Index setzen
df_seq2['datetime_utc'] = pd.to_datetime(df_seq2['DateTime(UTC)'], utc=True)
df_seq2 = df_seq2.set_index('datetime_utc')
df_seq2 = df_seq2[['Price[Currency/MWh]']].rename(columns={'Price[Currency/MWh]': 'price'})
df_seq2 = df_seq2.sort_index()

print("Shape:", df_seq2.shape)
print("Von:", df_seq2.index.min())
print("Bis:", df_seq2.index.max())
print("Fehlende Preise:", df_seq2['price'].isna().sum())

Shape: (284156, 1)
Von: 2018-09-30 22:00:00+00:00
Bis: 2026-04-21 21:45:00+00:00
Fehlende Preise: 0


In [7]:
# consolidated_full laden
df_main = pd.read_csv("raw_data/consolidated_full.csv",
                       index_col=0, parse_dates=True)

# Preis Spalte ersetzen
df_main = df_main.drop(columns=['price'])
df_merged = df_main.join(df_seq2, how='left')

print("Shape:", df_merged.shape)
print("Fehlende Preise:", df_merged['price'].isna().sum())
print(df_merged.head())

Shape: (284169, 10)
Fehlende Preise: 100
                           generation  consumption  wind_onshore  \
datetime_utc                                                       
2018-09-30 22:00:00+00:00    51434.81          NaN       4292.89   
2018-09-30 22:15:00+00:00    52085.57          NaN       4239.07   
2018-09-30 22:30:00+00:00    52345.33          NaN       4208.44   
2018-09-30 22:45:00+00:00    52902.03          NaN       4153.52   
2018-09-30 23:00:00+00:00    52799.83     42354.46       4047.26   

                           temperature_c  humidity_percent  \
datetime_utc                                                 
2018-09-30 22:00:00+00:00            NaN               NaN   
2018-09-30 22:15:00+00:00            NaN               NaN   
2018-09-30 22:30:00+00:00            NaN               NaN   
2018-09-30 22:45:00+00:00            NaN               NaN   
2018-09-30 23:00:00+00:00            NaN               NaN   

                           cloud_cover_percent 

In [9]:
df_merged.to_csv("raw_data/consolidated_seq2.csv")
print("✓ consolidated_seq2.csv gespeichert")
print("Shape:", df_merged.shape)
print("Fehlende Preise:", df_merged['price'].isna().sum())

✓ consolidated_seq2.csv gespeichert
Shape: (284169, 10)
Fehlende Preise: 100


In [10]:
df = pd.read_csv("raw_data/consolidated_seq2.csv",
                 index_col=0, parse_dates=True)

print("Shape:", df.shape)
print("Von:", df.index.min())
print("Bis:", df.index.max())
print("\nFehlende Werte:")
print(df.isnull().sum())
print("\nErste Zeilen:")
print(df.head())

Shape: (284169, 10)
Von: 2018-09-30 22:00:00+00:00
Bis: 2026-04-21 00:00:00+00:00

Fehlende Werte:
generation                     1
consumption                   17
wind_onshore                   2
temperature_c              86792
humidity_percent           86792
cloud_cover_percent        86796
shortwave_radiation_wm2    86796
wind_speed_ms              86792
ttf_gas                     6503
price                        100
dtype: int64

Erste Zeilen:
                           generation  consumption  wind_onshore  \
datetime_utc                                                       
2018-09-30 22:00:00+00:00    51434.81          NaN       4292.89   
2018-09-30 22:15:00+00:00    52085.57          NaN       4239.07   
2018-09-30 22:30:00+00:00    52345.33          NaN       4208.44   
2018-09-30 22:45:00+00:00    52902.03          NaN       4153.52   
2018-09-30 23:00:00+00:00    52799.83     42354.46       4047.26   

                           temperature_c  humidity_percent  \
date

In [1]:
import pandas as pd

df_main = pd.read_csv("raw_data/consolidated_full.csv", index_col=0, parse_dates=True)
df_gen = pd.read_csv("raw_data/generation_split.csv", index_col=0, parse_dates=True)

df_merged = df_main.join(df_gen, how='left')

print("Shape:", df_merged.shape)
print("Fehlende Renewable:", df_merged['generation_renewable'].isna().sum())

df_merged.to_csv("raw_data/consolidated_full.csv")
print("✓ consolidated_full.csv updated")

Shape: (265065, 12)
Fehlende Renewable: 9
✓ consolidated_full.csv updated


In [2]:
import pandas as pd

df_main = pd.read_csv("raw_data/consolidated_full.csv", index_col=0, parse_dates=True)
df_gen = pd.read_csv("raw_data/generation_split.csv", index_col=0, parse_dates=True)

print("consolidated_full:", df_main.shape)
print("generation_split:", df_gen.shape)

df_merged = df_main.join(df_gen, how='left')

print("\nShape nach merge:", df_merged.shape)
print("Fehlende generation_renewable:", df_merged['generation_renewable'].isna().sum())
print("Fehlende generation_non_renewable:", df_merged['generation_non_renewable'].isna().sum())

consolidated_full: (265065, 12)
generation_split: (265056, 2)


ValueError: columns overlap but no suffix specified: Index(['generation_renewable', 'generation_non_renewable'], dtype='object')

In [3]:
print(df_main.columns.tolist())

['generation', 'consumption', 'wind_onshore', 'temperature_c', 'humidity_percent', 'cloud_cover_percent', 'shortwave_radiation_wm2', 'wind_speed_ms', 'ttf_gas', 'price', 'generation_renewable', 'generation_non_renewable']


In [4]:
print(df_main[['generation_renewable', 'generation_non_renewable']].isna().sum())
print(df_main[['generation_renewable', 'generation_non_renewable']].head(10))

generation_renewable        9
generation_non_renewable    9
dtype: int64
                           generation_renewable  generation_non_renewable
datetime_utc                                                             
2018-09-30 22:00:00+00:00              11863.35                  28945.23
2018-09-30 22:15:00+00:00              11809.11                  29001.10
2018-09-30 22:30:00+00:00              11632.71                  29136.32
2018-09-30 22:45:00+00:00              11937.48                  28929.82
2018-09-30 23:00:00+00:00              12005.45                  28880.65
2018-09-30 23:15:00+00:00              11810.22                  28905.60
2018-09-30 23:30:00+00:00              11715.66                  28744.03
2018-09-30 23:45:00+00:00              11736.31                  28566.58
2018-10-01 00:00:00+00:00              11894.87                  28465.91
2018-10-01 00:15:00+00:00              11880.10                  28616.98


In [6]:
import pandas as pd

df = pd.read_csv("raw_data/consolidated_full.csv", index_col=0, parse_dates=True)
print("Shape:", df.shape)
print("Letztes Datum:", df.index.max())
print("Erstes Datum:", df.index.min())

Shape: (265065, 12)
Letztes Datum: 2026-04-23 00:00:00+00:00
Erstes Datum: 2018-09-30 22:00:00+00:00


In [7]:
import yfinance as yf
import pandas as pd

tickers = {'CL=F': 'wti_oil', 'BZ=F': 'brent_oil', 'NG=F': 'natural_gas'}

dfs = []
for ticker, col_name in tickers.items():
    data = yf.download(ticker, start='2018-10-01', end='2026-04-23', interval='1d', progress=False)
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)
    data = data[['Close']].rename(columns={'Close': col_name})
    if data.index.tz is None:
        data.index = data.index.tz_localize('UTC')
    data = data.resample('15min').ffill()
    data.index.name = 'datetime_utc'
    dfs.append(data)

df_commodities = pd.concat(dfs, axis=1)

# Mit consolidated_full mergen
df_main = pd.read_csv("raw_data/consolidated_full.csv", index_col=0, parse_dates=True)
df_merged = df_main.join(df_commodities, how='left')
df_merged.to_csv("raw_data/consolidated_full.csv")
print("✓ Shape:", df_merged.shape)
print("Spalten:", df_merged.columns.tolist())

✓ Shape: (265161, 15)
Spalten: ['generation', 'consumption', 'wind_onshore', 'temperature_c', 'humidity_percent', 'cloud_cover_percent', 'shortwave_radiation_wm2', 'wind_speed_ms', 'ttf_gas', 'price', 'generation_renewable', 'generation_non_renewable', 'wti_oil', 'brent_oil', 'natural_gas']


In [8]:
import openmeteo_requests
import requests_cache
from retry_requests import retry
import pandas as pd

# Setup
cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
om = openmeteo_requests.Client(session=retry_session)

# ERA5 Archive fetch
params = {
    'latitude': 51.1657,
    'longitude': 10.4515,
    'hourly': ['temperature_2m', 'relative_humidity_2m', 'cloud_cover',
               'shortwave_radiation', 'wind_speed_10m'],
    'start_date': '2018-10-01',
    'end_date': '2026-04-23',
    'timezone': 'UTC'
}

response = om.weather_api("https://archive-api.open-meteo.com/v1/archive", params=params)[0]
hourly = response.Hourly()

df_archive = pd.DataFrame({
    'temperature_c_observed':          hourly.Variables(0).ValuesAsNumpy(),
    'humidity_percent_observed':       hourly.Variables(1).ValuesAsNumpy(),
    'cloud_cover_percent_observed':    hourly.Variables(2).ValuesAsNumpy(),
    'shortwave_radiation_wm2_observed': hourly.Variables(3).ValuesAsNumpy(),
    'wind_speed_ms_observed':          hourly.Variables(4).ValuesAsNumpy()
}, index=pd.date_range(
    start=pd.to_datetime(hourly.Time(), unit='s', utc=True),
    end=pd.to_datetime(hourly.TimeEnd(), unit='s', utc=True),
    freq='h',
    inclusive='left'
))
df_archive.index.name = 'datetime_utc'
df_archive = df_archive.resample('15min').interpolate(method='linear')

print("Shape:", df_archive.shape)
print(df_archive.head())

Shape: (265149, 5)
                           temperature_c_observed  humidity_percent_observed  \
datetime_utc                                                                   
2018-10-01 00:00:00+00:00                  6.5500                  74.017944   
2018-10-01 00:15:00+00:00                  6.3375                  74.927086   
2018-10-01 00:30:00+00:00                  6.1250                  75.836227   
2018-10-01 00:45:00+00:00                  5.9125                  76.745377   
2018-10-01 01:00:00+00:00                  5.7000                  77.654518   

                           cloud_cover_percent_observed  \
datetime_utc                                              
2018-10-01 00:00:00+00:00                          3.00   
2018-10-01 00:15:00+00:00                          5.25   
2018-10-01 00:30:00+00:00                          7.50   
2018-10-01 00:45:00+00:00                          9.75   
2018-10-01 01:00:00+00:00                         12.00   

      

In [9]:
df_main = pd.read_csv("raw_data/consolidated_full.csv", index_col=0, parse_dates=True)

df_merged = df_main.join(df_archive, how='left')

print("Shape:", df_merged.shape)
print("Spalten:", df_merged.columns.tolist())
print("Fehlende observed:", df_merged['temperature_c_observed'].isna().sum())

df_merged.to_csv("raw_data/consolidated_full.csv")
print("✓ gespeichert")

Shape: (265161, 20)
Spalten: ['generation', 'consumption', 'wind_onshore', 'temperature_c', 'humidity_percent', 'cloud_cover_percent', 'shortwave_radiation_wm2', 'wind_speed_ms', 'ttf_gas', 'price', 'generation_renewable', 'generation_non_renewable', 'wti_oil', 'brent_oil', 'natural_gas', 'temperature_c_observed', 'humidity_percent_observed', 'cloud_cover_percent_observed', 'shortwave_radiation_wm2_observed', 'wind_speed_ms_observed']
Fehlende observed: 12
✓ gespeichert


In [3]:
import pandas as pd

df = pd.read_csv("raw_data/consolidated_full.csv", index_col=0, parse_dates=True)
today = pd.Timestamp.now(tz='UTC').normalize()

print("Letztes Datum:", df.index.max())
print("Heute:", today)
print("\nForecast Rows:")
print(df[df.index >= today][['temperature_c_forecast', 'humidity_percent_forecast']].head(20))

Letztes Datum: 2026-04-24 00:00:00+00:00
Heute: 2026-04-24 00:00:00+00:00

Forecast Rows:


KeyError: "None of [Index(['temperature_c_forecast', 'humidity_percent_forecast'], dtype='object')] are in the [columns]"

In [2]:
print(df.columns.tolist())

['generation', 'consumption', 'wind_onshore', 'temperature_c', 'humidity_percent', 'cloud_cover_percent', 'shortwave_radiation_wm2', 'wind_speed_ms', 'ttf_gas', 'price', 'generation_renewable', 'generation_non_renewable', 'wti_oil', 'brent_oil', 'natural_gas', 'temperature_c_observed', 'humidity_percent_observed', 'cloud_cover_percent_observed', 'shortwave_radiation_wm2_observed', 'wind_speed_ms_observed']


In [8]:
import pandas as pd

df = pd.read_csv("raw_data/consolidated_full.csv", index_col=0, parse_dates=True)
today = pd.Timestamp.now(tz='UTC').normalize()

print("Spalten:", df.columns.tolist())
print("Letztes Datum:", df.index.max())

print("\n=== Forecast Rows ===")
print(df[df.index >= today][['temperature_c_forecast', 'humidity_percent_forecast']].head(20))

Spalten: ['generation', 'consumption', 'wind_onshore', 'temperature_c', 'humidity_percent', 'cloud_cover_percent', 'shortwave_radiation_wm2', 'wind_speed_ms', 'ttf_gas', 'price', 'generation_renewable', 'generation_non_renewable', 'wti_oil', 'brent_oil', 'natural_gas', 'temperature_c_observed', 'humidity_percent_observed', 'cloud_cover_percent_observed', 'shortwave_radiation_wm2_observed', 'wind_speed_ms_observed']
Letztes Datum: 2026-04-24 00:00:00+00:00

=== Forecast Rows ===


KeyError: "None of [Index(['temperature_c_forecast', 'humidity_percent_forecast'], dtype='object')] are in the [columns]"

In [5]:
print(df.columns.tolist())

['generation', 'consumption', 'wind_onshore', 'temperature_c', 'humidity_percent', 'cloud_cover_percent', 'shortwave_radiation_wm2', 'wind_speed_ms', 'ttf_gas', 'price', 'generation_renewable', 'generation_non_renewable', 'wti_oil', 'brent_oil', 'natural_gas', 'temperature_c_observed', 'humidity_percent_observed', 'cloud_cover_percent_observed', 'shortwave_radiation_wm2_observed', 'wind_speed_ms_observed']


In [7]:
from fetcher import WeatherSource
import pandas as pd

ws = WeatherSource()
start = pd.Timestamp('2026-04-22', tz='UTC')
end = pd.Timestamp('2026-04-27', tz='UTC')

df_weather = ws.fetch(start, end)
print("Spalten:", df_weather.columns.tolist())
print("Letztes Datum:", df_weather.index.max())
print(df_weather.tail(10))

Spalten: ['temperature_c_observed', 'humidity_percent_observed', 'cloud_cover_percent_observed', 'shortwave_radiation_wm2_observed', 'wind_speed_ms_observed', 'temperature_c_forecast', 'humidity_percent_forecast', 'cloud_cover_percent_forecast', 'shortwave_radiation_wm2_forecast', 'wind_speed_ms_forecast']
Letztes Datum: 2026-04-27 23:00:00+00:00
                           temperature_c_observed  humidity_percent_observed  \
datetime_utc                                                                   
2026-04-27 20:45:00+00:00                     NaN                        NaN   
2026-04-27 21:00:00+00:00                     NaN                        NaN   
2026-04-27 21:15:00+00:00                     NaN                        NaN   
2026-04-27 21:30:00+00:00                     NaN                        NaN   
2026-04-27 21:45:00+00:00                     NaN                        NaN   
2026-04-27 22:00:00+00:00                     NaN                        NaN   
2026-04-27 

In [12]:
import pandas as pd

df = pd.read_csv("raw_data/consolidated_full.csv", index_col=0, parse_dates=True)
today = pd.Timestamp.now(tz='UTC').normalize()

print("Spalten:", df.columns.tolist())
print("Letztes Datum:", df.index.max())

print("\n=== Forecast Rows ===")
print(df[df.index >= today][['temperature_c_forecast', 'humidity_percent_forecast']].head(20))

Spalten: ['generation', 'consumption', 'wind_onshore', 'temperature_c', 'humidity_percent', 'cloud_cover_percent', 'shortwave_radiation_wm2', 'wind_speed_ms', 'ttf_gas', 'price', 'generation_renewable', 'generation_non_renewable', 'wti_oil', 'brent_oil', 'natural_gas', 'temperature_c_observed', 'humidity_percent_observed', 'cloud_cover_percent_observed', 'shortwave_radiation_wm2_observed', 'wind_speed_ms_observed', 'temperature_c_forecast', 'humidity_percent_forecast', 'cloud_cover_percent_forecast', 'shortwave_radiation_wm2_forecast', 'wind_speed_ms_forecast']
Letztes Datum: 2026-04-24 21:45:00+00:00

=== Forecast Rows ===
                           temperature_c_forecast  humidity_percent_forecast
datetime_utc                                                                
2026-04-24 00:00:00+00:00                  7.5000                      78.00
2026-04-24 00:15:00+00:00                  7.2125                      80.50
2026-04-24 00:30:00+00:00                  6.9250           

In [13]:
import pandas as pd
today = pd.Timestamp.now(tz='UTC').normalize()
df = pd.read_csv("raw_data/consolidated_full.csv", index_col=0, parse_dates=True)
print("Letztes Datum:", df.index.max())
print(df[df.index >= today][['temperature_c_forecast']].head(20))

Letztes Datum: 2026-04-24 21:45:00+00:00
                           temperature_c_forecast
datetime_utc                                     
2026-04-24 00:00:00+00:00                  7.5000
2026-04-24 00:15:00+00:00                  7.2125
2026-04-24 00:30:00+00:00                  6.9250
2026-04-24 00:45:00+00:00                  6.6375
2026-04-24 01:00:00+00:00                     NaN
2026-04-24 01:15:00+00:00                     NaN
2026-04-24 01:30:00+00:00                     NaN
2026-04-24 01:45:00+00:00                  5.7500
2026-04-24 02:00:00+00:00                     NaN
2026-04-24 02:15:00+00:00                     NaN
2026-04-24 02:30:00+00:00                     NaN
2026-04-24 02:45:00+00:00                     NaN
2026-04-24 03:00:00+00:00                  4.7500
2026-04-24 03:15:00+00:00                  4.6500
2026-04-24 03:30:00+00:00                  4.5500
2026-04-24 03:45:00+00:00                  4.4500
2026-04-24 04:00:00+00:00                     NaN
2026-04-2

In [14]:
import pandas as pd
df = pd.read_csv("raw_data/consolidated_full.csv", index_col=0, parse_dates=True)
print("Letztes Datum:", df.index.max())

Letztes Datum: 2026-04-24 21:45:00+00:00


In [15]:
import pandas as pd
df = pd.read_csv("raw_data/consolidated_full.csv", index_col=0, parse_dates=True)
print("Letztes Datum:", df.index.max())


Letztes Datum: 2026-04-27 23:00:00+00:00


In [16]:
import pandas as pd

df = pd.read_csv("raw_data/consolidated_full.csv", index_col=0, parse_dates=True)
print(df.columns.tolist())
print(df.shape)


['generation', 'consumption', 'wind_onshore', 'temperature_c', 'humidity_percent', 'cloud_cover_percent', 'shortwave_radiation_wm2', 'wind_speed_ms', 'ttf_gas', 'price', 'generation_renewable', 'generation_non_renewable', 'wti_oil', 'brent_oil', 'natural_gas', 'temperature_c_observed', 'humidity_percent_observed', 'cloud_cover_percent_observed', 'shortwave_radiation_wm2_observed', 'wind_speed_ms_observed', 'temperature_c_forecast', 'humidity_percent_forecast', 'cloud_cover_percent_forecast', 'shortwave_radiation_wm2_forecast', 'wind_speed_ms_forecast']
(265541, 25)


In [17]:
import pandas as pd

df = pd.read_csv("raw_data/consolidated_full.csv", index_col=0, parse_dates=True)

print("Index timezone:", df.index.tz)
print("\nErste Rows Index:")
print(df.index[:5])

print("\nLetzter bekannter Preis:")
print(df['price'].dropna().tail(3))

print("\nLetzter bekannter generation:")
print(df['generation'].dropna().tail(3))

print("\nLetzter bekannter temperature_c_observed:")
print(df['temperature_c_observed'].dropna().tail(3))

Index timezone: UTC

Erste Rows Index:
DatetimeIndex(['2018-09-30 22:00:00+00:00', '2018-09-30 22:15:00+00:00',
               '2018-09-30 22:30:00+00:00', '2018-09-30 22:45:00+00:00',
               '2018-09-30 23:00:00+00:00'],
              dtype='datetime64[ns, UTC]', name='datetime_utc', freq=None)

Letzter bekannter Preis:
datetime_utc
2026-04-24 21:15:00+00:00    114.80
2026-04-24 21:30:00+00:00    111.92
2026-04-24 21:45:00+00:00    109.09
Name: price, dtype: float64

Letzter bekannter generation:
datetime_utc
2026-04-24 08:45:00+00:00    67551.77690
2026-04-24 09:00:00+00:00    66989.05618
2026-04-24 09:15:00+00:00    68496.58525
Name: generation, dtype: float64

Letzter bekannter temperature_c_observed:
datetime_utc
2026-04-24 21:15:00+00:00    8.150001
2026-04-24 21:45:00+00:00    7.750000
2026-04-24 22:45:00+00:00    7.062500
Name: temperature_c_observed, dtype: float64


In [18]:
weather_cols = [c for c in df.columns if any(x in c for x in
               ['temperature', 'humidity', 'cloud', 'radiation', 'wind_speed'])]
print("Weather columns:", weather_cols)

df[weather_cols] = df[weather_cols].ffill()
df.to_csv("raw_data/consolidated_full.csv")
print("✓ Done")

Weather columns: ['temperature_c', 'humidity_percent', 'cloud_cover_percent', 'shortwave_radiation_wm2', 'wind_speed_ms', 'temperature_c_observed', 'humidity_percent_observed', 'cloud_cover_percent_observed', 'shortwave_radiation_wm2_observed', 'wind_speed_ms_observed', 'temperature_c_forecast', 'humidity_percent_forecast', 'cloud_cover_percent_forecast', 'shortwave_radiation_wm2_forecast', 'wind_speed_ms_forecast']
✓ Done


In [1]:
from google.cloud import bigquery
client = bigquery.Client(project='grid-intelligence-2026')
result = client.query('SELECT 1').result()
print('OK')

/Users/javierinocentezabala/.pyenv/versions/3.10.6/envs/grid-intelligence/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.6) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Forbidden: 403 POST https://bigquery.googleapis.com/bigquery/v2/projects/grid-intelligence-2026/jobs?prettyPrint=false: Access Denied: Project grid-intelligence-2026: User does not have bigquery.jobs.create permission in project grid-intelligence-2026.

Location: None
Job ID: 68f2be96-3bef-4d17-a3b7-f0f4ec8ec70e


In [10]:
import pandas as pd

df = pd.read_csv("raw_data/consolidated_full.csv", index_col=0, parse_dates=True)

print("Shape:", df.shape)
print("Von:", df.index.min())
print("Bis:", df.index.max())
print("\nFehlende Werte:")
print(df.isnull().sum())
print("\nFehlende Werte in %:")
print((df.isnull().sum() / len(df) * 100).round(2))

Shape: (265541, 25)
Von: 2018-09-30 22:00:00+00:00
Bis: 2026-04-27 23:00:00+00:00

Fehlende Werte:
generation                             342
consumption                            359
wind_onshore                           343
temperature_c                        87896
humidity_percent                     87896
cloud_cover_percent                  87900
shortwave_radiation_wm2              87900
wind_speed_ms                        87896
ttf_gas                               6522
price                                  393
generation_renewable                   342
generation_non_renewable               342
wti_oil                                437
brent_oil                              437
natural_gas                            437
temperature_c_observed                 371
humidity_percent_observed              371
cloud_cover_percent_observed           371
shortwave_radiation_wm2_observed       371
wind_speed_ms_observed                 371
temperature_c_forecast              26517

In [12]:
import pandas as pd
df = pd.read_csv('/Users/javierinocentezabala/code/grid-intelligence/raw_data/consolidated_full.csv', index_col=0, parse_dates=True, nrows=3)
print(df.shape)
print(df.columns.tolist())
print(df.head(2))


(3, 25)
['generation', 'consumption', 'wind_onshore', 'temperature_c', 'humidity_percent', 'cloud_cover_percent', 'shortwave_radiation_wm2', 'wind_speed_ms', 'ttf_gas', 'price', 'generation_renewable', 'generation_non_renewable', 'wti_oil', 'brent_oil', 'natural_gas', 'temperature_c_observed', 'humidity_percent_observed', 'cloud_cover_percent_observed', 'shortwave_radiation_wm2_observed', 'wind_speed_ms_observed', 'temperature_c_forecast', 'humidity_percent_forecast', 'cloud_cover_percent_forecast', 'shortwave_radiation_wm2_forecast', 'wind_speed_ms_forecast']
                           generation  consumption  wind_onshore  \
datetime_utc                                                       
2018-09-30 22:00:00+00:00    51434.81          NaN       4292.89   
2018-09-30 22:15:00+00:00    52085.57          NaN       4239.07   

                           temperature_c  humidity_percent  \
datetime_utc                                                 
2018-09-30 22:00:00+00:00           

In [13]:
import pandas as pd
df = pd.read_csv('/Users/javierinocentezabala/code/grid-intelligence/raw_data/consolidated_full.csv', index_col=0, parse_dates=True)
print('Shape:', df.shape)
print('Date range:', df.index.min(), '->', df.index.max())
print('Price NaN:', df['price'].isna().sum())
print('NaN per column:')
print(df.isna().sum())

Shape: (265829, 25)
Date range: 2018-09-30 22:00:00+00:00 -> 2026-04-30 23:00:00+00:00
Price NaN: 449
NaN per column:
generation                             428
consumption                            445
wind_onshore                           430
temperature_c                        88194
humidity_percent                     88194
cloud_cover_percent                  88198
shortwave_radiation_wm2              88198
wind_speed_ms                        88194
ttf_gas                               6608
price                                  449
generation_renewable                   428
generation_non_renewable               428
wti_oil                                523
brent_oil                              523
natural_gas                            523
temperature_c_observed                 430
humidity_percent_observed              430
cloud_cover_percent_observed           430
shortwave_radiation_wm2_observed       430
wind_speed_ms_observed                 430
temperature_c_forecast

In [15]:
from grid_intelligence.logic.preprocessor import generate_features

df = generate_features(nrows=10000, train=True)
print(df.shape)
print(df.columns.tolist())

/Users/javierinocentezabala/code/grid-intelligence/grid_intelligence/logic/data.py:95: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns, UTC] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  df["is_holiday"] = df[datetime_col].dt.floor("D").isin(de_holidays)
/Users/javierinocentezabala/code/grid-intelligence/grid_intelligence/logic/data.py:124: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns, UTC] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  df['is_holiday_288'] = future_timestamp.dt.floor("D").isin(de_holidays).astype(int)


(7031, 165)
['generation', 'consumption', 'wind_onshore', 'ttf_gas', 'price', 'generation_renewable', 'generation_non_renewable', 'wti_oil', 'natural_gas', 'temperature_c_observed', 'cloud_cover_percent_observed', 'shortwave_radiation_wm2_observed', 'wind_speed_ms_observed', 'quarter_hour_sin', 'quarter_hour_cos', 'hour_sin', 'hour_cos', 'day_of_week_sin', 'day_of_week_cos', 'day_of_year_sin', 'day_of_year_cos', 'month_sin', 'month_cos', 'price_lag_1', 'price_lag_4', 'price_lag_12', 'price_lag_24', 'price_lag_96', 'price_lag_672', 'price_roll_mean_24', 'price_roll_mean_96', 'price_roll_mean_672', 'price_roll_std_4', 'price_roll_std_96', 'generation_renewable_lag_1', 'generation_renewable_lag_4', 'generation_renewable_lag_24', 'generation_renewable_lag_96', 'generation_renewable_lag_672', 'generation_renewable_roll_mean_24', 'generation_renewable_roll_mean_96', 'generation_renewable_roll_mean_672', 'generation_renewable_roll_std_24', 'generation_renewable_roll_std_96', 'generation_non_r

In [ ]:
import pickle
import numpy as np
from pathlib import Path
from xgboost import XGBClassifier, XGBRegressor
from sklearn.metrics import mean_absolute_error

# ── Target erstellen ──────────────────────────────────────────────────────────
df['target_288'] = df['price'].shift(-288)
df = df.dropna(subset=['target_288', 'price']).reset_index(drop=True)
print(f"Nach target: {df.shape}")

# ── Regime definieren ─────────────────────────────────────────────────────────
threshold_pos = 150.0
threshold_neg = 0.0

df['regime'] = np.where(df['target_288'] > threshold_pos, 1,
               np.where(df['target_288'] < threshold_neg, 2, 0))
print("Regime Verteilung:")
print(df['regime'].value_counts())

# ── Features / Target ─────────────────────────────────────────────────────────
drop_cols = ['price', 'target_288', 'regime']
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols]
y_reg = df['target_288']
y_clf = df['regime']

split_idx = int(len(df) * 0.85)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_reg_train, y_reg_test = y_reg.iloc[:split_idx], y_reg.iloc[split_idx:]
y_clf_train, y_clf_test = y_clf.iloc[:split_idx], y_clf.iloc[split_idx:]

# ── Training ──────────────────────────────────────────────────────────────────
print("\nTraining...")

regime_classifier = XGBClassifier(n_estimators=100, max_depth=4, random_state=42, n_jobs=-1, eval_metric='mlogloss')
regime_classifier.fit(X_train, y_clf_train)

model_normal = XGBRegressor(n_estimators=100, max_depth=4, random_state=42, n_jobs=-1)
model_normal.fit(X_train[y_clf_train == 0], y_reg_train[y_clf_train == 0])

model_pos = XGBRegressor(n_estimators=100, max_depth=4, random_state=42, n_jobs=-1)
mask_pos = y_clf_train == 1
if mask_pos.sum() > 0:
    model_pos.fit(X_train[mask_pos], y_reg_train[mask_pos])
else:
    model_pos.fit(X_train, y_reg_train)  # fallback

model_neg = XGBRegressor(n_estimators=100, max_depth=4, random_state=42, n_jobs=-1)
mask_neg = y_clf_train == 2
if mask_neg.sum() > 0:
    model_neg.fit(X_train[mask_neg], y_reg_train[mask_neg])
else:
    model_neg.fit(X_train, y_reg_train)  # fallback

# ── MAE ───────────────────────────────────────────────────────────────────────
probs = regime_classifier.predict_proba(X_test)
p_normal, p_pos, p_neg = probs[:, 0], probs[:, 1], probs[:, 2]
final_preds = (p_normal * model_normal.predict(X_test) +
               p_pos    * model_pos.predict(X_test) +
               p_neg    * model_neg.predict(X_test))
mae = mean_absolute_error(y_reg_test, final_preds)
print(f"MAE: {mae:.2f} EUR/MWh")

# ── Speichern ─────────────────────────────────────────────────────────────────
models_dir = Path('/Users/javierinocentezabala/code/grid-intelligence/grid_intelligence/models')
models_dir.mkdir(exist_ok=True)

model_config = {
    'threshold_pos': threshold_pos,
    'threshold_neg': threshold_neg,
    'scaling_factors': {'normal': 1.0, 'pos': 1.0, 'neg': 1.0},
    'feature_cols': feature_cols,
    'n_features': len(feature_cols),
    'overall_mae': mae,
}

for name, obj in [
    ('regime_classifier', regime_classifier),
    ('model_normal',      model_normal),
    ('model_pos',         model_pos),
    ('model_neg',         model_neg),
    ('model_config',      model_config),
]:
    with open(models_dir / f'{name}.pkl', 'wb') as f:
        pickle.dump(obj, f)
    print(f"✅ Saved: {name}.pkl")


Nach target: (6743, 166)
Regime Verteilung:
regime
0    5881
1     621
2     241
Name: count, dtype: int64

Training...
MAE: 33.78 EUR/MWh
✅ Saved: regime_classifier.pkl
✅ Saved: model_normal.pkl
✅ Saved: model_pos.pkl
✅ Saved: model_neg.pkl
✅ Saved: model_config.pkl


In [17]:
from grid_intelligence.logic.preprocessor import generate_features
df = generate_features(nrows=1632, train=False)
print(df.columns.tolist()[:5])
print(df.index.name)

/Users/javierinocentezabala/code/grid-intelligence/grid_intelligence/logic/data.py:95: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns, UTC] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  df["is_holiday"] = df[datetime_col].dt.floor("D").isin(de_holidays)
/Users/javierinocentezabala/code/grid-intelligence/grid_intelligence/logic/data.py:124: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns, UTC] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  df['is_holiday_288'] = future_timestamp.dt.floor("D").isin(de_holidays).astype(int)


['generation', 'consumption', 'wind_onshore', 'ttf_gas', 'generation_renewable']
None


In [18]:
import pandas as pd
df = pd.read_csv('/Users/javierinocentezabala/code/grid-intelligence/raw_data/consolidated_full.csv')
df['datetime_utc'] = pd.to_datetime(df['datetime_utc'], utc=True)
df['target_288'] = df['price'].shift(-288)
df['hour'] = df['datetime_utc'].dt.hour
print(df.groupby('hour')['target_288'].mean().round(2))


hour
0      77.98
1      75.52
2      75.31
3      79.68
4      92.25
5     106.03
6     111.63
7     105.39
8      95.22
9      85.99
10     78.22
11     71.79
12     70.15
13     74.56
14     85.47
15    101.79
16    120.87
17    132.56
18    129.31
19    116.12
20    104.64
21     95.31
22     87.03
23     82.00
Name: target_288, dtype: float64


In [21]:
from grid_intelligence.logic.preprocessor import generate_features
df = generate_features(nrows=30000, train=True)
print(df.shape)

/Users/javierinocentezabala/code/grid-intelligence/grid_intelligence/logic/data.py:95: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns, UTC] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  df["is_holiday"] = df[datetime_col].dt.floor("D").isin(de_holidays)
/Users/javierinocentezabala/code/grid-intelligence/grid_intelligence/logic/data.py:124: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns, UTC] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  df['is_holiday_288'] = future_timestamp.dt.floor("D").isin(de_holidays).astype(int)


(23961, 165)


In [23]:
import pickle
from pathlib import Path
from xgboost import XGBClassifier, XGBRegressor
from sklearn.metrics import mean_absolute_error

# Target
df['target_288'] = df['price'].shift(-288)
df = df.dropna(subset=['target_288', 'price']).reset_index(drop=True)
print(f"Nach target: {df.shape}")

# Regime
threshold_pos = 150.0
threshold_neg = 0.0
df['regime'] = np.where(df['target_288'] > threshold_pos, 1,
               np.where(df['target_288'] < threshold_neg, 2, 0))
print("Regime:", df['regime'].value_counts().to_dict())

# Features
drop_cols = ['price', 'target_288', 'regime', 'datetime_utc']
feature_cols = [c for c in df.columns if c not in drop_cols]
X = df[feature_cols]
y_reg = df['target_288']
y_clf = df['regime']

# Split
split_idx = int(len(df) * 0.85)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_reg_train, y_reg_test = y_reg.iloc[:split_idx], y_reg.iloc[split_idx:]
y_clf_train, y_clf_test = y_clf.iloc[:split_idx], y_clf.iloc[split_idx:]
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# Train
print("Training...")
xgb_params = dict(n_estimators=300, max_depth=6, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1)

regime_classifier = XGBClassifier(**xgb_params, eval_metric='mlogloss')
regime_classifier.fit(X_train, y_clf_train)

model_normal = XGBRegressor(**xgb_params)
model_normal.fit(X_train[y_clf_train==0], y_reg_train[y_clf_train==0])

model_pos = XGBRegressor(**xgb_params)
model_pos.fit(X_train[y_clf_train==1], y_reg_train[y_clf_train==1]) if (y_clf_train==1).sum() > 0 else model_pos.fit(X_train, y_reg_train)

model_neg = XGBRegressor(**xgb_params)
model_neg.fit(X_train[y_clf_train==2], y_reg_train[y_clf_train==2]) if (y_clf_train==2).sum() > 0 else model_neg.fit(X_train, y_reg_train)

# MAE
probs = regime_classifier.predict_proba(X_test)
final_preds = probs[:,0]*model_normal.predict(X_test) + probs[:,1]*model_pos.predict(X_test) + probs[:,2]*model_neg.predict(X_test)
mae = mean_absolute_error(y_reg_test, final_preds)
print(f"MAE: {mae:.2f} EUR/MWh")

# Save
models_dir = Path('/Users/javierinocentezabala/code/grid-intelligence/grid_intelligence/models')
models_dir.mkdir(exist_ok=True)
model_config = {'threshold_pos': threshold_pos, 'threshold_neg': threshold_neg,
                'scaling_factors': {'normal': 1.0, 'pos': 1.0, 'neg': 1.0},
                'feature_cols': feature_cols, 'n_features': len(feature_cols), 'overall_mae': mae}

for name, obj in [('regime_classifier', regime_classifier), ('model_normal', model_normal),
                  ('model_pos', model_pos), ('model_neg', model_neg), ('model_config', model_config)]:
    with open(models_dir / f'{name}.pkl', 'wb') as f:
        pickle.dump(obj, f)
    print(f"✅ Saved: {name}.pkl")

Nach target: (23673, 166)
Regime: {0: 21134, 1: 1718, 2: 821}
Train: (20122, 164), Test: (3551, 164)
Training...
MAE: 39.70 EUR/MWh
✅ Saved: regime_classifier.pkl
✅ Saved: model_normal.pkl
✅ Saved: model_pos.pkl
✅ Saved: model_neg.pkl
✅ Saved: model_config.pkl


In [24]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from xgboost import XGBClassifier, XGBRegressor
from sklearn.metrics import mean_absolute_error
import sys
sys.path.insert(0, '/Users/javierinocentezabala/code/grid-intelligence')
from grid_intelligence.logic.data import add_time, add_lag, add_rolling_mean, add_rolling_std
from grid_intelligence.logic.preprocessor import add_interaction_features

# Load data
df = pd.read_csv('/Users/javierinocentezabala/code/grid-intelligence/raw_data/consolidated_full.csv')
df['datetime_utc'] = pd.to_datetime(df['datetime_utc'], utc=True)
df = df[df['datetime_utc'] <= pd.Timestamp.now(tz='UTC')]
df = df.tail(5000).reset_index(drop=True)

# Forward fill commodities
for col in ['ttf_gas', 'wti_oil', 'brent_oil', 'natural_gas']:
    if col in df.columns:
        df[col] = df[col].ffill()

# Convert to Berlin time for time features
df['datetime_berlin'] = df['datetime_utc'].dt.tz_convert('Europe/Berlin')
df = add_time(df, datetime_col="datetime_berlin")
df = df.drop(columns=['datetime_berlin', 'datetime_utc'], errors='ignore')

print(df.shape)
print("hour_sin last 5:", df['hour_sin'].tail(5).values)

/Users/javierinocentezabala/code/grid-intelligence/grid_intelligence/logic/data.py:95: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns, Europe/Berlin] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  df["is_holiday"] = df[datetime_col].dt.floor("D").isin(de_holidays)
/Users/javierinocentezabala/code/grid-intelligence/grid_intelligence/logic/data.py:124: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns, Europe/Berlin] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  df['is_holiday_288'] = future_timestamp.dt.floor("D").isin(de_holidays).astype(int)


(5000, 40)
hour_sin last 5: [1.         1.         1.         1.         0.96592583]


In [25]:
# Add lags
df = add_lag(df, target_col="price")
df = add_rolling_mean(df, target_col="price")
df = add_rolling_std(df, target_col="price")
df = add_lag(df, target_col="generation_renewable")
df = add_rolling_mean(df, target_col="generation_renewable")
df = add_lag(df, target_col="consumption")
df = add_rolling_mean(df, target_col="consumption")
df = add_lag(df, target_col="ttf_gas")
df = add_lag(df, target_col="wti_oil")
df = add_interaction_features(df)

# Target
df['target_288'] = df['price'].shift(-288)
df = df.dropna(subset=['target_288', 'price']).reset_index(drop=True)
print(f"Shape: {df.shape}")

# Features
drop_cols = ['price', 'target_288']
feature_cols = [c for c in df.columns if c not in drop_cols]
X = df[feature_cols]
y = df['target_288']

# Train
split_idx = int(len(df) * 0.85)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

model = XGBRegressor(n_estimators=100, max_depth=4, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
mae = mean_absolute_error(y_test, model.predict(X_test))
print(f"MAE: {mae:.2f} EUR/MWh")

# Check peak hour
preds = model.predict(X_test)
df_test = X_test.copy()
df_test['pred'] = preds
print("Avg prediction by hour_sin:")
print(df_test.groupby(df_test['hour_sin'].round(2))['pred'].mean().sort_values(ascending=False).head(5))

Shape: (4692, 93)
MAE: 45.34 EUR/MWh
Avg prediction by hour_sin:
hour_sin
 1.00    134.539795
 0.97    130.309540
-1.00    125.213348
-0.97    120.741928
 0.87    118.742737
Name: pred, dtype: float32


In [26]:
from grid_intelligence.logic.preprocessor import generate_features

df = generate_features(nrows=300000, train=True)
print(df.shape)
print('hour_sin avg by price bucket:')
import pandas as pd
df['price_bucket'] = pd.cut(df['price'], bins=[0, 50, 100, 150, 300])
print(df.groupby('price_bucket')['hour_sin'].mean())

/Users/javierinocentezabala/code/grid-intelligence/grid_intelligence/logic/data.py:95: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns, UTC] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  df["is_holiday"] = df[datetime_col].dt.floor("D").isin(de_holidays)
/Users/javierinocentezabala/code/grid-intelligence/grid_intelligence/logic/data.py:124: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns, UTC] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  df['is_holiday_288'] = future_timestamp.dt.floor("D").isin(de_holidays).astype(int)


(235202, 165)
hour_sin avg by price bucket:
price_bucket
(0, 50]       0.053170
(50, 100]     0.014509
(100, 150]   -0.076306
(150, 300]   -0.092646
Name: hour_sin, dtype: float64


/var/folders/6p/0b4w1hy94rg79cl81qv1csjc0000gn/T/ipykernel_93044/1004019189.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby('price_bucket')['hour_sin'].mean())


In [27]:
print('datetime_utc dtype:', df['datetime_utc'].dtype if 'datetime_utc' in df.columns else 'dropped')
print('Columns with datetime:', [c for c in df.columns if 'datetime' in c.lower()])

datetime_utc dtype: dropped
Columns with datetime: []


In [31]:
import os
os.environ['ENV'] = 'development'

import importlib
import grid_intelligence.logic.preprocessor as pp
import grid_intelligence.params as params
importlib.reload(params)
importlib.reload(pp)
from grid_intelligence.logic.preprocessor import generate_features

df = generate_features(nrows=50000, train=True)
print(df.shape)
print('hour_sin avg by price bucket:')
import pandas as pd
df['price_bucket'] = pd.cut(df['price'], bins=[0, 50, 100, 150, 300])
print(df.groupby('price_bucket', observed=True)['hour_sin'].mean())

/Users/javierinocentezabala/code/grid-intelligence/grid_intelligence/logic/data.py:95: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns, Europe/Berlin] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  df["is_holiday"] = df[datetime_col].dt.floor("D").isin(de_holidays)
/Users/javierinocentezabala/code/grid-intelligence/grid_intelligence/logic/data.py:124: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns, Europe/Berlin] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  df['is_holiday_288'] = future_timestamp.dt.floor("D").isin(de_holidays).astype(int)


(49745, 166)
hour_sin avg by price bucket:
price_bucket
(0, 50]       0.007049
(50, 100]     0.160021
(100, 150]   -0.074001
(150, 300]   -0.285576
Name: hour_sin, dtype: float64


In [ ]:
import pickle
from pathlib import Path
from xgboost import XGBClassifier, XGBRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

# Target
df['target_288'] = df['price'].shift(-288)
df = df.dropna(subset=['target_288', 'price']).reset_index(drop=True)
print(f"Shape: {df.shape}")

# Regime
threshold_pos = 150.0
threshold_neg = 0.0
df['regime'] = np.where(df['target_288'] > threshold_pos, 1,
               np.where(df['target_288'] < threshold_neg, 2, 0))
print("Regime:", df['regime'].value_counts().to_dict())

# Features
drop_cols = ['price', 'target_288', 'regime', 'datetime_utc', 'price_bucket']
feature_cols = [c for c in df.columns if c not in drop_cols]
X = df[feature_cols]
y_reg = df['target_288']
y_clf = df['regime']

split_idx = int(len(df) * 0.85)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_reg_train, y_reg_test = y_reg.iloc[:split_idx], y_reg.iloc[split_idx:]
y_clf_train, y_clf_test = y_clf.iloc[:split_idx], y_clf.iloc[split_idx:]

xgb_params = dict(n_estimators=300, max_depth=6, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1)

print("Training...")
regime_classifier = XGBClassifier(**xgb_params, eval_metric='mlogloss')
regime_classifier.fit(X_train, y_clf_train)

model_normal = XGBRegressor(**xgb_params)
model_normal.fit(X_train[y_clf_train==0], y_reg_train[y_clf_train==0])

model_pos = XGBRegressor(**xgb_params)
mask_pos = y_clf_train==1
model_pos.fit(X_train[mask_pos], y_reg_train[mask_pos]) if mask_pos.sum()>0 else model_pos.fit(X_train, y_reg_train)

model_neg = XGBRegressor(**xgb_params)
mask_neg = y_clf_train==2
model_neg.fit(X_train[mask_neg], y_reg_train[mask_neg]) if mask_neg.sum()>0 else model_neg.fit(X_train, y_reg_train)

probs = regime_classifier.predict_proba(X_test)
final_preds = probs[:,0]*model_normal.predict(X_test) + probs[:,1]*model_pos.predict(X_test) + probs[:,2]*model_neg.predict(X_test)
mae = mean_absolute_error(y_reg_test, final_preds)
print(f"MAE: {mae:.2f} EUR/MWh")

# Save
models_dir = Path('/Users/javierinocentezabala/code/grid-intelligence/grid_intelligence/models')
models_dir.mkdir(exist_ok=True)
model_config = {'threshold_pos': threshold_pos, 'threshold_neg': threshold_neg,
                'scaling_factors': {'normal': 1.0, 'pos': 1.0, 'neg': 1.0},
                'feature_cols': feature_cols, 'n_features': len(feature_cols), 'overall_mae': mae}

for name, obj in [('regime_classifier', regime_classifier), ('model_normal', model_normal),
                  ('model_pos', model_pos), ('model_neg', model_neg), ('model_config', model_config)]:
    with open(models_dir / f'{name}.pkl', 'wb') as f:
        pickle.dump(obj, f)
    print(f"✅ {name}.pkl")

Shape: (49457, 168)
Regime: {0: 41838, 1: 4777, 2: 2842}
Training...


ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:price_bucket: category